# Human–AI Convention Change Model

This notebook is generated from the tested Python script.

Run cells in order. Edit the data path in the settings cell if needed.


In [ ]:
# ---------------------------------------------------------------
# JUPYTER SETTINGS — edit these, then run all cells in order
# ---------------------------------------------------------------
import sys
from pathlib import Path

DATA_DIR = Path.home() / "Downloads" / "name_game_20 2"
OUT_DIR = DATA_DIR / "python_model_outputs"
STARTS = 12
PPC = 500
SEED = 20260818

sys.argv = [
    "name_game_abcd_model",
    "--data-dir", str(DATA_DIR),
    "--out-dir", str(OUT_DIR),
    "--starts", str(STARTS),
    "--ppc", str(PPC),
    "--seed", str(SEED),
]

assert DATA_DIR.is_dir(), f"Data directory not found: {DATA_DIR}"
%matplotlib inline

print("Data dir:", DATA_DIR)
print("Out dir: ", OUT_DIR)


In [ ]:
#!/usr/bin/env python3
"""
Name Game: Human-AI convention-change models A-D
================================================

Post-change rounds: 8-14
Payoffs: J=10, M=20, N=20

Latent candidate conventions:
    G_J: preserve J
    G_M: transition toward M
    G_N: transition toward N

Prior:
    P(G_J)=1-h
    P(G_M)=P(G_N)=h/2

Switching response:
    p_switch(A) is logistic in inferred advantage A and is parameterized
    by its endpoint probabilities:
        p_lo = P(switch | A=-10)
        p_hi = P(switch | A=20)

Models:
    A: shared h, shared response curve
    B: Human/AI h differ, shared response curve
    C: shared h, Human/AI response curves differ
    D: Human/AI h and response curves differ

Likelihood:
    Actor-marginal likelihood. Each observed action is predicted by
    sum_G q(G) P(action | G), followed by public Bayesian updating.

Lapse:
    epsilon is uniform over 26 letters:
      J/M/N each get epsilon/26
      Other aggregates 23*epsilon/26

Requires:
    numpy pandas scipy matplotlib jax jaxlib

Example:
    python name_game_abcd_model.py \
      --data-dir "$HOME/Downloads/name_game_20 2"
"""

from __future__ import annotations

import argparse
import json
import os
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

os.environ.setdefault("JAX_ENABLE_X64", "true")

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import expit, logit
import matplotlib.pyplot as plt

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax.scipy.special import logsumexp


# ---------------------------------------------------------------------
# Fixed experiment quantities
# ---------------------------------------------------------------------

ROUNDS = tuple(range(8, 15))
GROUP_SIZE = 6
PAYOFF_J = 10.0
PAYOFF_MN = 20.0
A_LOW = -10.0
A_HIGH = 20.0
ACTION_NAMES = ("J", "M", "N", "Other")
TYPE_NAMES = ("Human", "AI")
MODELS = ("A", "B", "C", "D")
N_PARAMETERS = {"A": 4, "B": 5, "C": 6, "D": 7}

# Numerical guardrails, not substantive priors.
H_LO, H_HI = 0.005, 0.95
P_LO_MIN, P_LO_MAX = 0.02, 0.97
P_HI_MAX = 0.98
EPS_LO, EPS_HI = 1e-5, 0.20


@dataclass(frozen=True)
class Group:
    key: str
    n_human: int
    n_ai: int
    counts: np.ndarray  # shape: (7 rounds, 2 types, 4 action categories)


# ---------------------------------------------------------------------
# Data loading
# ---------------------------------------------------------------------

def find_input_files(data_dir: Path) -> Tuple[Path, Path]:
    """Find either the local Downloads names or ChatGPT-upload names."""
    candidates = [
        (
            "all_apps_wide-2026-08-05-2.csv",
            "all_apps_wide-2026-08-05-2_part2.csv",
        ),
        (
            "all_apps_wide-2026-08-05-2(1).csv",
            "all_apps_wide-2026-08-05-2_part2(1).csv",
        ),
    ]
    for a, b in candidates:
        p1, p2 = data_dir / a, data_dir / b
        if p1.exists() and p2.exists():
            return p1, p2
    raise FileNotFoundError(
        f"Could not find the two expected CSV files in: {data_dir}"
    )


def action_code(x: object) -> int:
    x = str(x).strip().upper()
    return {"J": 0, "M": 1, "N": 2}.get(x, 3)


def first_parseable_json_list(values: Sequence[object]) -> List[str]:
    """Group-level AI JSON is duplicated across human rows; parse one valid copy."""
    for value in values:
        if pd.isna(value):
            continue
        text = str(value).strip()
        if not text or text == "PENDING":
            continue
        try:
            parsed = json.loads(text)
        except Exception:
            continue
        if isinstance(parsed, list):
            return [str(x) for x in parsed]
    return []


def load_groups(file1: Path, file2: Path) -> Tuple[List[Group], pd.DataFrame, pd.DataFrame]:
    raw = pd.concat(
        [
            pd.read_csv(file1, low_memory=False),
            pd.read_csv(file2, low_memory=False),
        ],
        ignore_index=True,
    ).drop_duplicates()

    # Known excluded/incomplete session from the experiment workflow.
    raw = raw.loc[raw["session.code"] != "un9nmcsy"].copy()

    gid_col = "name_game.8.group.id_in_subsession"
    candidate = raw.loc[raw[gid_col].notna()].copy()

    groups: List[Group] = []
    retained_rows = []
    excluded_rows = []

    for (session, gid), g in candidate.groupby(["session.code", gid_col], sort=False):
        reason = None

        try:
            n_human = int(g["name_game.8.group.num_humans"].dropna().iloc[0])
            n_ai = int(g["name_game.8.group.num_llms"].dropna().iloc[0])
        except Exception:
            reason = "missing composition"
            n_human = n_ai = -1

        if reason is None and n_human + n_ai != GROUP_SIZE:
            reason = "composition does not sum to six"

        # This script is specifically for the mixed-team dataset.
        if reason is None and (n_human <= 0 or n_ai <= 0):
            reason = "not a mixed Human-AI group"

        if reason is None:
            humans = g.loc[g["name_game.8.player.letter_choice"].notna()].copy()
            humans = humans.drop_duplicates("participant.code")
            humans = humans.sort_values("name_game.8.player.id_in_group")
            if len(humans) != n_human:
                reason = f"human count mismatch ({len(humans)} vs {n_human})"

        if reason is None:
            counts = np.zeros((len(ROUNDS), 2, 4), dtype=np.int64)

            for round_index, r in enumerate(ROUNDS):
                human_values = humans[f"name_game.{r}.player.letter_choice"].tolist()
                if len(human_values) != n_human or any(pd.isna(x) for x in human_values):
                    reason = f"incomplete human choices in round {r}"
                    break

                ai_values = first_parseable_json_list(
                    g[f"name_game.{r}.group.llm_choices_json"].tolist()
                )
                if len(ai_values) != n_ai:
                    reason = f"AI count mismatch in round {r} ({len(ai_values)} vs {n_ai})"
                    break

                for x in human_values:
                    counts[round_index, 0, action_code(x)] += 1
                for x in ai_values:
                    counts[round_index, 1, action_code(x)] += 1

            if reason is None:
                if not np.all(counts.sum(axis=(1, 2)) == GROUP_SIZE):
                    reason = "round total != six"
                elif not np.all(counts[:, 0].sum(axis=1) == n_human):
                    reason = "human round totals inconsistent"
                elif not np.all(counts[:, 1].sum(axis=1) == n_ai):
                    reason = "AI round totals inconsistent"

        key = f"{session}_G{int(gid)}"
        if reason is not None:
            excluded_rows.append(
                {
                    "group": key,
                    "session": session,
                    "group_id": int(gid),
                    "reason": reason,
                }
            )
            continue

        groups.append(Group(key, n_human, n_ai, counts))
        retained_rows.append(
            {
                "group": key,
                "session": session,
                "group_id": int(gid),
                "n_human": n_human,
                "n_ai": n_ai,
            }
        )

    if not groups:
        raise RuntimeError("No complete mixed groups were reconstructed.")

    retained = pd.DataFrame(retained_rows)
    excluded = pd.DataFrame(excluded_rows)
    return groups, retained, excluded


# ---------------------------------------------------------------------
# Parameter transforms
# ---------------------------------------------------------------------

def bounded_np(z: float, lo: float, hi: float) -> float:
    return float(lo + (hi - lo) * expit(z))


def inverse_bounded(x: float, lo: float, hi: float) -> float:
    p = np.clip((x - lo) / (hi - lo), 1e-10, 1 - 1e-10)
    return float(logit(p))


def bounded_jax(z, lo: float, hi: float):
    return lo + (hi - lo) * jax.nn.sigmoid(z)


def curve_jax(z_lo, z_delta):
    p_lo = bounded_jax(z_lo, P_LO_MIN, P_LO_MAX)
    fraction = bounded_jax(z_delta, 0.001, 0.999)
    p_hi = p_lo + fraction * (P_HI_MAX - p_lo)
    return p_lo, p_hi


def transform_jax(theta, model: str):
    i = 0

    if model == "A":
        hH = hA = bounded_jax(theta[i], H_LO, H_HI)
        i += 1
        p_loH, p_hiH = curve_jax(theta[i], theta[i + 1])
        i += 2
        p_loA, p_hiA = p_loH, p_hiH

    elif model == "B":
        hH = bounded_jax(theta[i], H_LO, H_HI)
        hA = bounded_jax(theta[i + 1], H_LO, H_HI)
        i += 2
        p_loH, p_hiH = curve_jax(theta[i], theta[i + 1])
        i += 2
        p_loA, p_hiA = p_loH, p_hiH

    elif model == "C":
        hH = hA = bounded_jax(theta[i], H_LO, H_HI)
        i += 1
        p_loH, p_hiH = curve_jax(theta[i], theta[i + 1])
        i += 2
        p_loA, p_hiA = curve_jax(theta[i], theta[i + 1])
        i += 2

    elif model == "D":
        hH = bounded_jax(theta[i], H_LO, H_HI)
        hA = bounded_jax(theta[i + 1], H_LO, H_HI)
        i += 2
        p_loH, p_hiH = curve_jax(theta[i], theta[i + 1])
        i += 2
        p_loA, p_hiA = curve_jax(theta[i], theta[i + 1])
        i += 2

    else:
        raise ValueError(f"Unknown model: {model}")

    epsilon = bounded_jax(theta[i], EPS_LO, EPS_HI)

    return jnp.array(
        [hH, hA, p_loH, p_hiH, p_loA, p_hiA, epsilon],
        dtype=jnp.float64,
    )


def parameters_to_dict(values: np.ndarray) -> Dict[str, float]:
    names = ("hH", "hA", "p_loH", "p_hiH", "p_loA", "p_hiA", "epsilon")
    return {name: float(value) for name, value in zip(names, values)}


def decode_np(theta: np.ndarray, model: str) -> Dict[str, float]:
    return parameters_to_dict(
        np.asarray(transform_jax(jnp.asarray(theta, dtype=jnp.float64), model), dtype=float)
    )


def curve_inverse(p_lo: float, p_hi: float) -> Tuple[float, float]:
    z_lo = inverse_bounded(p_lo, P_LO_MIN, P_LO_MAX)
    fraction = np.clip((p_hi - p_lo) / (P_HI_MAX - p_lo), 0.001, 0.999)
    z_delta = inverse_bounded(fraction, 0.001, 0.999)
    return z_lo, z_delta


def initial_theta(model: str) -> np.ndarray:
    eps = inverse_bounded(0.04, EPS_LO, EPS_HI)

    if model == "A":
        lo, d = curve_inverse(0.70, 0.88)
        return np.array([inverse_bounded(0.35, H_LO, H_HI), lo, d, eps])

    if model == "B":
        lo, d = curve_inverse(0.70, 0.88)
        return np.array(
            [
                inverse_bounded(0.50, H_LO, H_HI),
                inverse_bounded(0.18, H_LO, H_HI),
                lo,
                d,
                eps,
            ]
        )

    if model == "C":
        hlo, hd = curve_inverse(0.84, 0.89)
        alo, ad = curve_inverse(0.55, 0.88)
        return np.array(
            [inverse_bounded(0.35, H_LO, H_HI), hlo, hd, alo, ad, eps]
        )

    if model == "D":
        hlo, hd = curve_inverse(0.84, 0.89)
        alo, ad = curve_inverse(0.55, 0.88)
        return np.array(
            [
                inverse_bounded(0.50, H_LO, H_HI),
                inverse_bounded(0.18, H_LO, H_HI),
                hlo,
                hd,
                alo,
                ad,
                eps,
            ]
        )

    raise ValueError(model)


def encode_parameters(p: Dict[str, float], model: str) -> np.ndarray:
    eps = inverse_bounded(p["epsilon"], EPS_LO, EPS_HI)

    if model == "A":
        h = (p["hH"] + p["hA"]) / 2
        plo = (p["p_loH"] + p["p_loA"]) / 2
        phi = (p["p_hiH"] + p["p_hiA"]) / 2
        lo, d = curve_inverse(plo, phi)
        return np.array([inverse_bounded(h, H_LO, H_HI), lo, d, eps])

    if model == "B":
        plo = (p["p_loH"] + p["p_loA"]) / 2
        phi = (p["p_hiH"] + p["p_hiA"]) / 2
        lo, d = curve_inverse(plo, phi)
        return np.array(
            [
                inverse_bounded(p["hH"], H_LO, H_HI),
                inverse_bounded(p["hA"], H_LO, H_HI),
                lo,
                d,
                eps,
            ]
        )

    if model == "C":
        h = (p["hH"] + p["hA"]) / 2
        hlo, hd = curve_inverse(p["p_loH"], p["p_hiH"])
        alo, ad = curve_inverse(p["p_loA"], p["p_hiA"])
        return np.array(
            [inverse_bounded(h, H_LO, H_HI), hlo, hd, alo, ad, eps]
        )

    if model == "D":
        hlo, hd = curve_inverse(p["p_loH"], p["p_hiH"])
        alo, ad = curve_inverse(p["p_loA"], p["p_hiA"])
        return np.array(
            [
                inverse_bounded(p["hH"], H_LO, H_HI),
                inverse_bounded(p["hA"], H_LO, H_HI),
                hlo,
                hd,
                alo,
                ad,
                eps,
            ]
        )

    raise ValueError(model)


# ---------------------------------------------------------------------
# JAX likelihood
# ---------------------------------------------------------------------

def action_matrix_jax(q, p_lo, p_hi, epsilon):
    advantage_M = PAYOFF_MN * q[1] - PAYOFF_J * q[0]
    advantage_N = PAYOFF_MN * q[2] - PAYOFF_J * q[0]

    logit_lo = jnp.log(p_lo / (1.0 - p_lo))
    logit_hi = jnp.log(p_hi / (1.0 - p_hi))
    beta = (logit_hi - logit_lo) / (A_HIGH - A_LOW)
    alpha = logit_lo - beta * A_LOW

    adopt_M = jax.nn.sigmoid(alpha + beta * advantage_M)
    adopt_N = jax.nn.sigmoid(alpha + beta * advantage_N)

    lapse_one = epsilon / 26.0
    lapse_other = 23.0 * lapse_one
    intentional = 1.0 - epsilon

    return jnp.array(
        [
            [intentional + lapse_one, lapse_one, lapse_one, lapse_other],
            [
                intentional * (1.0 - adopt_M) + lapse_one,
                intentional * adopt_M + lapse_one,
                lapse_one,
                lapse_other,
            ],
            [
                intentional * (1.0 - adopt_N) + lapse_one,
                lapse_one,
                intentional * adopt_N + lapse_one,
                lapse_other,
            ],
        ],
        dtype=jnp.float64,
    )


def group_loglik_jax(counts, parameters):
    hH, hA, p_loH, p_hiH, p_loA, p_hiA, epsilon = parameters

    qH = jnp.array([1.0 - hH, hH / 2.0, hH / 2.0], dtype=jnp.float64)
    qA = jnp.array([1.0 - hA, hA / 2.0, hA / 2.0], dtype=jnp.float64)

    ll = 0.0

    for r in range(len(ROUNDS)):
        human_emissions = action_matrix_jax(qH, p_loH, p_hiH, epsilon)
        ai_emissions = action_matrix_jax(qA, p_loA, p_hiA, epsilon)

        pred_human = qH @ human_emissions
        pred_ai = qA @ ai_emissions

        ll += jnp.sum(counts[r, 0] * jnp.log(jnp.maximum(pred_human, 1e-300)))
        ll += jnp.sum(counts[r, 1] * jnp.log(jnp.maximum(pred_ai, 1e-300)))

        # Public evidence likelihood by candidate policy.
        log_likelihood = jnp.sum(
            counts[r, 0][None, :]
            * jnp.log(jnp.maximum(human_emissions, 1e-300)),
            axis=1,
        )
        log_likelihood += jnp.sum(
            counts[r, 1][None, :]
            * jnp.log(jnp.maximum(ai_emissions, 1e-300)),
            axis=1,
        )

        zH = jnp.log(jnp.maximum(qH, 1e-300)) + log_likelihood
        qH = jnp.exp(zH - logsumexp(zH))

        zA = jnp.log(jnp.maximum(qA, 1e-300)) + log_likelihood
        qA = jnp.exp(zA - logsumexp(zA))

    return ll


def make_objective(model: str):
    def total_nll(theta, counts):
        parameters = transform_jax(theta, model)
        group_ll = jax.vmap(group_loglik_jax, in_axes=(0, None))(counts, parameters)
        return -jnp.sum(group_ll)

    value_and_grad = jax.jit(jax.value_and_grad(total_nll))
    value_only = jax.jit(total_nll)
    return value_and_grad, value_only


# ---------------------------------------------------------------------
# Fitting
# ---------------------------------------------------------------------

def fit_model(
    model: str,
    counts: np.ndarray,
    n_starts: int,
    seed: int,
    warm_starts: List[np.ndarray] | None = None,
):
    value_and_grad, _ = make_objective(model)
    jax_counts = jnp.asarray(counts, dtype=jnp.float64)

    base = initial_theta(model)

    # Compile once before timing optimizer starts.
    compiled_value, _ = value_and_grad(jnp.asarray(base), jax_counts)
    compiled_value.block_until_ready()

    def objective(theta):
        value, gradient = value_and_grad(jnp.asarray(theta), jax_counts)
        return float(value), np.asarray(gradient, dtype=float)

    rng = np.random.default_rng(seed)
    starts = [np.asarray(x, dtype=float) for x in (warm_starts or [])]
    starts.append(base)

    while len(starts) < n_starts:
        starts.append(base + rng.normal(0.0, 0.9, size=base.size))

    best = None
    for start in starts:
        result = minimize(
            objective,
            start,
            jac=True,
            method="L-BFGS-B",
            options={
                "maxiter": 2000,
                "ftol": 1e-12,
                "gtol": 1e-8,
                "maxls": 50,
            },
        )
        if best is None or result.fun < best.fun:
            best = result

    if best is None or not np.isfinite(best.fun):
        raise RuntimeError(f"All optimizer starts failed for Model {model}")

    return best, decode_np(best.x, model)


# ---------------------------------------------------------------------
# Independent NumPy likelihood used for self-test and simulation
# ---------------------------------------------------------------------

def switch_probability_np(advantage: float, p_lo: float, p_hi: float) -> float:
    beta = (logit(p_hi) - logit(p_lo)) / (A_HIGH - A_LOW)
    alpha = logit(p_lo) - beta * A_LOW
    return float(expit(alpha + beta * advantage))


def action_matrix_np(q: np.ndarray, p_lo: float, p_hi: float, epsilon: float) -> np.ndarray:
    q = np.asarray(q, dtype=float)
    advantage_M = PAYOFF_MN * q[1] - PAYOFF_J * q[0]
    advantage_N = PAYOFF_MN * q[2] - PAYOFF_J * q[0]
    adopt_M = switch_probability_np(advantage_M, p_lo, p_hi)
    adopt_N = switch_probability_np(advantage_N, p_lo, p_hi)

    lapse_one = epsilon / 26.0
    lapse_other = 23.0 * lapse_one
    intentional = 1.0 - epsilon

    matrix = np.array(
        [
            [intentional + lapse_one, lapse_one, lapse_one, lapse_other],
            [intentional * (1-adopt_M) + lapse_one, intentional*adopt_M + lapse_one, lapse_one, lapse_other],
            [intentional * (1-adopt_N) + lapse_one, lapse_one, intentional*adopt_N + lapse_one, lapse_other],
        ],
        dtype=float,
    )

    if not np.isfinite(matrix).all():
        raise ValueError("Non-finite action probability")
    if np.any(matrix <= 0):
        raise ValueError("Action probabilities must be positive")
    if not np.allclose(matrix.sum(axis=1), 1.0, atol=1e-10):
        raise ValueError("Action probabilities do not sum to one")
    return matrix


def update_beliefs_np(qH, qA, round_counts, p):
    human_emissions = action_matrix_np(qH, p["p_loH"], p["p_hiH"], p["epsilon"])
    ai_emissions = action_matrix_np(qA, p["p_loA"], p["p_hiA"], p["epsilon"])

    log_likelihood = (
        round_counts[0][None, :] * np.log(human_emissions)
    ).sum(axis=1)
    log_likelihood += (
        round_counts[1][None, :] * np.log(ai_emissions)
    ).sum(axis=1)

    def update(q):
        z = np.log(np.maximum(q, 1e-300)) + log_likelihood
        z -= np.max(z)
        out = np.exp(z)
        return out / out.sum()

    return update(qH), update(qA)


def group_loglik_np(group: Group, p: Dict[str, float]) -> float:
    qH = np.array([1-p["hH"], p["hH"]/2, p["hH"]/2], dtype=float)
    qA = np.array([1-p["hA"], p["hA"]/2, p["hA"]/2], dtype=float)
    ll = 0.0

    for r in range(len(ROUNDS)):
        H = action_matrix_np(qH, p["p_loH"], p["p_hiH"], p["epsilon"])
        A = action_matrix_np(qA, p["p_loA"], p["p_hiA"], p["epsilon"])
        predH = qH @ H
        predA = qA @ A
        ll += float(group.counts[r, 0] @ np.log(predH))
        ll += float(group.counts[r, 1] @ np.log(predA))
        qH, qA = update_beliefs_np(qH, qA, group.counts[r], p)

    return ll


def self_tests(groups: List[Group], counts: np.ndarray):
    assert PAYOFF_J == 10.0
    assert PAYOFF_MN == 20.0
    assert counts.shape == (len(groups), 7, 2, 4)
    assert np.all(counts.sum(axis=(2, 3)) == GROUP_SIZE)

    for model in MODELS:
        theta = initial_theta(model)
        p = decode_np(theta, model)
        assert p["p_hiH"] >= p["p_loH"]
        assert p["p_hiA"] >= p["p_loA"]

        q = np.array([1-p["hH"], p["hH"]/2, p["hH"]/2])
        matrix = action_matrix_np(q, p["p_loH"], p["p_hiH"], p["epsilon"])
        assert np.allclose(matrix.sum(axis=1), 1.0)
        assert np.isfinite(group_loglik_np(groups[0], p))

        _, value_only = make_objective(model)
        jax_nll = float(value_only(jnp.asarray(theta), jnp.asarray(counts)))
        numpy_nll = -sum(group_loglik_np(g, p) for g in groups)
        if not np.isclose(jax_nll, numpy_nll, atol=1e-8, rtol=1e-10):
            raise AssertionError(
                f"JAX/NumPy likelihood mismatch for Model {model}: "
                f"{jax_nll} vs {numpy_nll}"
            )


# ---------------------------------------------------------------------
# Predictive simulation
# ---------------------------------------------------------------------

def simulate_group(group: Group, p: Dict[str, float], rng: np.random.Generator) -> np.ndarray:
    qH = np.array([1-p["hH"], p["hH"]/2, p["hH"]/2], dtype=float)
    qA = np.array([1-p["hA"], p["hA"]/2, p["hA"]/2], dtype=float)
    counts = np.zeros((7, 2, 4), dtype=np.int64)

    for r in range(7):
        H = action_matrix_np(qH, p["p_loH"], p["p_hiH"], p["epsilon"])
        A = action_matrix_np(qA, p["p_loA"], p["p_hiA"], p["epsilon"])
        predH = qH @ H
        predA = qA @ A

        counts[r, 0] = rng.multinomial(group.n_human, predH)
        counts[r, 1] = rng.multinomial(group.n_ai, predA)
        qH, qA = update_beliefs_np(qH, qA, counts[r], p)

    return counts


def switching_ppc(groups: List[Group], p: Dict[str, float], n_sim: int, seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    rows = []

    for simulation in range(n_sim):
        numerator = np.zeros((7, 2), dtype=float)
        denominator = np.zeros((7, 2), dtype=float)

        for group in groups:
            sim = simulate_group(group, p, rng)
            numerator += sim[:, :, 1] + sim[:, :, 2]
            denominator += sim.sum(axis=2)

        rates = numerator / denominator
        for r_index, r in enumerate(ROUNDS):
            for type_index, type_name in enumerate(TYPE_NAMES):
                rows.append(
                    {
                        "simulation": simulation,
                        "round": r,
                        "type": type_name,
                        "switch_rate": rates[r_index, type_index],
                    }
                )

    raw = pd.DataFrame(rows)
    return (
        raw.groupby(["round", "type"], as_index=False)
        .agg(
            simulated_mean=("switch_rate", "mean"),
            lower_95=("switch_rate", lambda x: np.quantile(x, 0.025)),
            upper_95=("switch_rate", lambda x: np.quantile(x, 0.975)),
        )
    )


def observed_switching(groups: List[Group]) -> pd.DataFrame:
    rows = []
    for r_index, r in enumerate(ROUNDS):
        for type_index, type_name in enumerate(TYPE_NAMES):
            numerator = sum(
                g.counts[r_index, type_index, 1] + g.counts[r_index, type_index, 2]
                for g in groups
            )
            denominator = sum(g.counts[r_index, type_index].sum() for g in groups)
            rows.append(
                {
                    "round": r,
                    "type": type_name,
                    "switch_rate": numerator / denominator,
                    "n": int(denominator),
                }
            )
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------
# Diagnostics / plotting
# ---------------------------------------------------------------------

def boundary_flags(p: Dict[str, float]) -> List[str]:
    flags = []
    if p["hH"] < H_LO + 0.01 or p["hH"] > H_HI - 0.01:
        flags.append("hH near bound")
    if p["hA"] < H_LO + 0.01 or p["hA"] > H_HI - 0.01:
        flags.append("hA near bound")
    if p["p_loH"] < P_LO_MIN + 0.01 or p["p_loH"] > P_LO_MAX - 0.01:
        flags.append("p_loH near bound")
    if p["p_loA"] < P_LO_MIN + 0.01 or p["p_loA"] > P_LO_MAX - 0.01:
        flags.append("p_loA near bound")
    if p["p_hiH"] > P_HI_MAX - 0.01:
        flags.append("p_hiH near upper bound")
    if p["p_hiA"] > P_HI_MAX - 0.01:
        flags.append("p_hiA near upper bound")
    return flags


def plot_switching(observed: pd.DataFrame, ppc: pd.DataFrame, output: Path):
    fig, ax = plt.subplots(figsize=(7.5, 4.8))

    for type_name in TYPE_NAMES:
        sim = ppc.loc[ppc["type"] == type_name]
        obs = observed.loc[observed["type"] == type_name]
        line = ax.plot(
            sim["round"],
            sim["simulated_mean"],
            marker="o",
            label=f"{type_name} simulated",
        )[0]
        ax.fill_between(
            sim["round"],
            sim["lower_95"],
            sim["upper_95"],
            alpha=0.15,
            color=line.get_color(),
        )
        ax.plot(
            obs["round"],
            obs["switch_rate"],
            marker="x",
            linestyle="--",
            color=line.get_color(),
            label=f"{type_name} observed",
        )

    ax.set_ylim(0, 1)
    ax.set_xticks(ROUNDS)
    ax.set_xlabel("Round")
    ax.set_ylabel("Proportion choosing M or N")
    ax.set_title("Model D predictive check")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(output, dpi=180)
    plt.show()


def plot_response_curves(p: Dict[str, float], output: Path):
    advantages = np.linspace(A_LOW, A_HIGH, 300)
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    for label, lo_key, hi_key in (
        ("Human", "p_loH", "p_hiH"),
        ("AI", "p_loA", "p_hiA"),
    ):
        probs = [
            switch_probability_np(a, p[lo_key], p[hi_key])
            for a in advantages
        ]
        ax.plot(advantages, probs, label=label)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Inferred transition advantage")
    ax.set_ylabel("Switch probability")
    ax.set_title("Model D response curves")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(output, dpi=180)
    plt.show()


# ---------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--data-dir",
        type=Path,
        default=Path.home() / "Downloads" / "name_game_20 2",
    )
    parser.add_argument("--out-dir", type=Path, default=None)
    parser.add_argument("--starts", type=int, default=12)
    parser.add_argument("--ppc", type=int, default=500)
    parser.add_argument("--seed", type=int, default=20260818)
    args = parser.parse_args()

    if args.starts < 3:
        raise ValueError("Use at least 3 optimization starts for the main analysis.")
    if args.ppc < 0:
        raise ValueError("--ppc must be >= 0")

    file1, file2 = find_input_files(args.data_dir)
    output_dir = args.out_dir or (args.data_dir / "python_model_outputs")
    output_dir.mkdir(parents=True, exist_ok=True)

    start_time = time.perf_counter()

    groups, retained, excluded = load_groups(file1, file2)
    counts = np.stack([g.counts for g in groups]).astype(np.float64)
    n_observations = len(groups) * GROUP_SIZE * len(ROUNDS)

    print(f"Loaded {len(groups)} complete mixed groups; {n_observations} choices.")
    print(retained.groupby(["n_human", "n_ai"]).size().rename("groups").to_string())

    # Current two pilot exports should reconstruct exactly this dataset.
    if len(groups) != 18 or n_observations != 756:
        print(
            "WARNING: current reference exports previously reconstructed as "
            "18 groups / 756 choices. Inspect retained_groups.csv and exclusions.csv."
        )

    retained.to_csv(output_dir / "retained_groups.csv", index=False)
    excluded.to_csv(output_dir / "exclusions.csv", index=False)

    self_tests(groups, counts)
    print("Self-tests passed, including independent NumPy/JAX likelihood agreement.")

    fits = {}
    comparison_rows = []

    for model in MODELS:
        warm = []
        if model == "B" and "A" in fits:
            warm.append(encode_parameters(fits["A"][1], "B"))
        elif model == "C" and "A" in fits:
            warm.append(encode_parameters(fits["A"][1], "C"))
        elif model == "D":
            warm.append(encode_parameters(fits["B"][1], "D"))
            warm.append(encode_parameters(fits["C"][1], "D"))

        t_model = time.perf_counter()
        result, parameters = fit_model(
            model,
            counts,
            n_starts=args.starts,
            seed=args.seed + ord(model),
            warm_starts=warm,
        )
        fits[model] = (result, parameters)

        k = N_PARAMETERS[model]
        aic = 2 * result.fun + 2 * k
        bic = 2 * result.fun + k * np.log(n_observations)
        flags = boundary_flags(parameters)

        comparison_rows.append(
            {
                "model": model,
                "NLL": result.fun,
                "k": k,
                "AIC": aic,
                "BIC": bic,
                "optimizer_success": bool(result.success),
                "optimizer_message": str(result.message),
                "boundary_flags": "; ".join(flags),
                **parameters,
            }
        )

        print(
            f"Model {model}: NLL={result.fun:.3f}, AIC={aic:.3f}, "
            f"BIC={bic:.3f}, time={time.perf_counter()-t_model:.2f}s"
        )
        if flags:
            print("  Boundary diagnostic:", ", ".join(flags))

    comparison = pd.DataFrame(comparison_rows)
    comparison["delta_AIC"] = comparison["AIC"] - comparison["AIC"].min()
    comparison["delta_BIC"] = comparison["BIC"] - comparison["BIC"].min()

    # Nested-model sanity checks.
    nlls = comparison.set_index("model")["NLL"]
    if nlls["B"] > nlls["A"] + 1e-5:
        raise RuntimeError("Model B fit is worse than nested Model A; increase --starts.")
    if nlls["C"] > nlls["A"] + 1e-5:
        raise RuntimeError("Model C fit is worse than nested Model A; increase --starts.")
    if nlls["D"] > min(nlls["B"], nlls["C"]) + 1e-5:
        raise RuntimeError("Model D fit is worse than a nested model; increase --starts.")

    print("\nMODEL COMPARISON")
    print(
        comparison[
            [
                "model", "NLL", "AIC", "BIC", "delta_AIC", "delta_BIC",
                "hH", "hA", "p_loH", "p_hiH", "p_loA", "p_hiA", "epsilon",
                "boundary_flags",
            ]
        ].to_string(index=False)
    )

    comparison.to_csv(output_dir / "model_comparison_and_parameters.csv", index=False)

    observed = observed_switching(groups)
    observed.to_csv(output_dir / "observed_switching.csv", index=False)

    # Model D response curve output.
    pD = fits["D"][1]
    response_rows = []
    for advantage in np.linspace(A_LOW, A_HIGH, 301):
        response_rows.append(
            {
                "advantage": advantage,
                "Human": switch_probability_np(advantage, pD["p_loH"], pD["p_hiH"]),
                "AI": switch_probability_np(advantage, pD["p_loA"], pD["p_hiA"]),
            }
        )
    pd.DataFrame(response_rows).to_csv(output_dir / "model_D_response_curves.csv", index=False)
    plot_response_curves(pD, output_dir / "model_D_response_curves.png")

    if args.ppc > 0:
        t_ppc = time.perf_counter()
        ppc = switching_ppc(groups, pD, args.ppc, args.seed + 999)
        ppc.to_csv(output_dir / "model_D_switching_ppc.csv", index=False)
        plot_switching(observed, ppc, output_dir / "model_D_switching_ppc.png")
        print(
            f"Model D PPC ({args.ppc} simulated datasets): "
            f"{time.perf_counter()-t_ppc:.2f}s"
        )

    print(f"\nFinished in {time.perf_counter()-start_time:.2f}s")
    print(f"Outputs: {output_dir}")

    return {
        "groups": groups,
        "counts": counts,
        "retained": retained,
        "excluded": excluded,
        "fits": fits,
        "comparison": comparison,
        "observed": observed,
        "output_dir": output_dir,
        "args": args,
    }


if __name__ == "__main__":
    RUN = main()


## Validation settings

These are deliberately separate from the main fit so you can change the cost of the validation analyses without touching the core model.

- Cross-validation is **whole-group**: no group contributes observations to both train and test sets.
- Parameter recovery generates varied, interior Model-D parameter settings and refits D without initializing at the generating values.
- Model recovery simulates from the empirical fitted A–D models and fits all four candidate models to each synthetic dataset.


In [ ]:
# Validation controls
SIMULATION_DRAWS = 500

CV_FOLDS = 6
CV_REPEATS = 3
VALIDATION_STARTS = 5

PARAM_RECOVERY_REPS = 30
PARAM_RECOVERY_THETA_SD = 0.8

MODEL_RECOVERY_REPS = 15

VALIDATION_SEED = SEED + 50000

print("Validation settings:")
print("  simulation draws:", SIMULATION_DRAWS)
print("  CV:", CV_FOLDS, "folds x", CV_REPEATS, "repeats")
print("  parameter-recovery datasets:", PARAM_RECOVERY_REPS)
print("  model-recovery datasets per generating model:", MODEL_RECOVERY_REPS)


## Validation helpers


In [ ]:
from functools import lru_cache

PARAMETER_NAMES = ("hH", "hA", "p_loH", "p_hiH", "p_loA", "p_hiA", "epsilon")

_validation_objective_cache = {}

def fit_model_cached(
    model,
    counts,
    n_starts,
    seed,
    warm_starts=None,
):
    """Fit a model while reusing compiled JAX objectives of the same shape."""
    key = (model, int(counts.shape[0]))

    if key not in _validation_objective_cache:
        _validation_objective_cache[key] = make_objective(model)

    value_and_grad, _ = _validation_objective_cache[key]
    jax_counts = jnp.asarray(counts, dtype=jnp.float64)
    base = initial_theta(model)

    # Trigger JIT compilation once for this model / training-set size.
    compiled_value, _ = value_and_grad(jnp.asarray(base), jax_counts)
    compiled_value.block_until_ready()

    def objective_local(theta):
        value, gradient = value_and_grad(
            jnp.asarray(theta, dtype=jnp.float64),
            jax_counts,
        )
        return float(value), np.asarray(gradient, dtype=float)

    rng = np.random.default_rng(seed)
    starts = [np.asarray(x, dtype=float) for x in (warm_starts or [])]
    starts.append(base.copy())

    while len(starts) < n_starts:
        starts.append(
            base + rng.normal(0.0, 0.9, size=base.size)
        )

    best = None

    for start in starts:
        result = minimize(
            objective_local,
            start,
            jac=True,
            method="L-BFGS-B",
            options={
                "maxiter": 2000,
                "ftol": 1e-12,
                "gtol": 1e-8,
                "maxls": 50,
            },
        )

        if np.isfinite(result.fun) and (
            best is None or result.fun < best.fun
        ):
            best = result

    if best is None:
        raise RuntimeError(f"All validation fits failed for Model {model}")

    return best, decode_np(best.x, model)


def fit_all_models_validation(counts, n_starts, seed):
    """Fit A-D with nesting-preserving warm starts within one dataset."""
    out = {}

    for model in MODELS:
        warm = []

        if model == "B":
            warm.append(encode_parameters(out["A"][1], "B"))
        elif model == "C":
            warm.append(encode_parameters(out["A"][1], "C"))
        elif model == "D":
            warm.extend([
                encode_parameters(out["B"][1], "D"),
                encode_parameters(out["C"][1], "D"),
            ])

        out[model] = fit_model_cached(
            model=model,
            counts=counts,
            n_starts=n_starts,
            seed=seed + ord(model),
            warm_starts=warm,
        )

    # Every more flexible model must fit at least as well as its nested parent.
    nll = {m: out[m][0].fun for m in MODELS}

    if nll["B"] > nll["A"] + 1e-5:
        raise RuntimeError("Validation Model B fit worse than nested A.")
    if nll["C"] > nll["A"] + 1e-5:
        raise RuntimeError("Validation Model C fit worse than nested A.")
    if nll["D"] > min(nll["B"], nll["C"]) + 1e-5:
        raise RuntimeError("Validation Model D fit worse than nested B/C.")

    return out


def simulate_groups_from_parameters(template_groups, parameters, rng):
    """Simulate a complete synthetic dataset with the observed group compositions."""
    synthetic = []

    for group in template_groups:
        synthetic_counts = simulate_group(group, parameters, rng)

        synthetic.append(
            Group(
                key=group.key,
                n_human=group.n_human,
                n_ai=group.n_ai,
                counts=synthetic_counts,
            )
        )

    return synthetic


def stack_group_counts(groups_use):
    return np.stack([g.counts for g in groups_use]).astype(np.float64)


## Simulation plot

This is an in-notebook predictive simulation check for Model D. The band is the central 95% of simulated aggregate M/N adoption rates; the solid simulated mean is compared with the observed trajectory.


In [ ]:
groups = RUN["groups"]
fits = RUN["fits"]
output_dir = RUN["output_dir"]

pD = fits["D"][1]

simulation_ppc = switching_ppc(
    groups,
    pD,
    n_sim=SIMULATION_DRAWS,
    seed=VALIDATION_SEED,
)

simulation_ppc.to_csv(
    output_dir / "model_D_simulation_plot_data.csv",
    index=False,
)

for type_name in TYPE_NAMES:
    sim = simulation_ppc.loc[
        simulation_ppc["type"] == type_name
    ].sort_values("round")

    obs = RUN["observed"].loc[
        RUN["observed"]["type"] == type_name
    ].sort_values("round")

    fig, ax = plt.subplots(figsize=(6.4, 4.2))

    ax.fill_between(
        sim["round"].to_numpy(),
        sim["lower_95"].to_numpy(),
        sim["upper_95"].to_numpy(),
        alpha=0.18,
        label="95% simulated interval",
    )

    ax.plot(
        sim["round"],
        sim["simulated_mean"],
        marker="o",
        linewidth=1.8,
        label="Simulated mean",
    )

    ax.plot(
        obs["round"],
        obs["switch_rate"],
        marker="s",
        linestyle="--",
        linewidth=1.8,
        label="Observed",
    )

    ax.set(
        xlabel="Round",
        ylabel="Proportion choosing M or N",
        ylim=(0, 1),
        xticks=ROUNDS,
        title=f"Model D predictive simulation: {type_name}",
    )
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(
        output_dir / f"model_D_simulation_{type_name.lower()}.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()


## Whole-group cross-validation

Each fold holds out entire six-player groups. Models are estimated only on the remaining groups, and the held-out log likelihood is then evaluated with no updating from the held-out data before prediction beyond the model's ordinary within-group sequential updating.

The principal comparison is total held-out negative log likelihood; lower is better. Because every model is evaluated on exactly the same held-out choices, this directly compares out-of-sample predictive accuracy without an AIC/BIC complexity penalty.


In [ ]:
def make_group_folds(n_groups, k, repeats, seed):
    if k < 2 or k > n_groups:
        raise ValueError("CV_FOLDS must be between 2 and the number of groups.")

    rng = np.random.default_rng(seed)
    folds = []

    for repeat in range(repeats):
        permutation = rng.permutation(n_groups)
        test_chunks = np.array_split(permutation, k)

        for fold, test_idx in enumerate(test_chunks):
            train_mask = np.ones(n_groups, dtype=bool)
            train_mask[test_idx] = False
            train_idx = np.flatnonzero(train_mask)

            folds.append(
                (repeat + 1, fold + 1, train_idx, np.asarray(test_idx))
            )

    return folds


def run_group_cross_validation(
    groups,
    k=6,
    repeats=3,
    n_starts=5,
    seed=20260818,
):
    all_counts = stack_group_counts(groups)
    rows = []

    fold_definitions = make_group_folds(
        len(groups),
        k,
        repeats,
        seed,
    )

    for repeat, fold, train_idx, test_idx in fold_definitions:
        train_counts = all_counts[train_idx]

        fold_fits = fit_all_models_validation(
            train_counts,
            n_starts=n_starts,
            seed=seed + repeat * 10000 + fold * 100,
        )

        n_test_choices = (
            len(test_idx) * GROUP_SIZE * len(ROUNDS)
        )

        for model in MODELS:
            result, parameters = fold_fits[model]

            test_loglik = sum(
                group_loglik_np(groups[i], parameters)
                for i in test_idx
            )

            rows.append({
                "repeat": repeat,
                "fold": fold,
                "model": model,
                "n_train_groups": len(train_idx),
                "n_test_groups": len(test_idx),
                "n_test_choices": n_test_choices,
                "train_NLL": float(result.fun),
                "test_loglik": float(test_loglik),
                "test_NLL": float(-test_loglik),
                "test_NLL_per_choice": float(-test_loglik / n_test_choices),
                "optimizer_success": bool(result.success),
            })

    raw = pd.DataFrame(rows)

    summary = (
        raw.groupby("model", as_index=False)
        .agg(
            total_test_NLL=("test_NLL", "sum"),
            mean_test_NLL_per_choice=("test_NLL_per_choice", "mean"),
            sd_test_NLL_per_choice=("test_NLL_per_choice", "std"),
            folds=("test_NLL", "size"),
        )
        .sort_values("total_test_NLL")
        .reset_index(drop=True)
    )

    summary["delta_total_test_NLL"] = (
        summary["total_test_NLL"] -
        summary["total_test_NLL"].min()
    )

    return raw, summary


cv_raw, cv_summary = run_group_cross_validation(
    groups=groups,
    k=CV_FOLDS,
    repeats=CV_REPEATS,
    n_starts=VALIDATION_STARTS,
    seed=VALIDATION_SEED + 1000,
)

print("WHOLE-GROUP CROSS-VALIDATION")
display(cv_summary)

cv_raw.to_csv(
    output_dir / "group_cross_validation_folds.csv",
    index=False,
)
cv_summary.to_csv(
    output_dir / "group_cross_validation_summary.csv",
    index=False,
)

fig, ax = plt.subplots(figsize=(6.2, 4.0))
plot_dat = cv_summary.sort_values("total_test_NLL")

ax.bar(
    plot_dat["model"],
    plot_dat["delta_total_test_NLL"],
)
ax.set(
    xlabel="Model",
    ylabel="Δ held-out NLL",
    title="Whole-group cross-validation",
)
fig.tight_layout()
fig.savefig(
    output_dir / "group_cross_validation.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


## Model-D parameter recovery

For each synthetic dataset, a new valid Model-D parameter vector is generated by perturbing an interior point in unconstrained parameter space. This deliberately varies the true parameters across datasets rather than recovering one fixed setting.

The fitted model is **not** initialized at the generating parameter vector. Recovery therefore tests whether the data identify the parameters rather than whether the optimizer can return to a supplied answer.


In [ ]:
def run_parameter_recovery_D(
    template_groups,
    n_reps=30,
    n_starts=5,
    theta_sd=0.8,
    seed=20260818,
):
    rng = np.random.default_rng(seed)
    base_theta = initial_theta("D")
    rows = []

    for rep in range(1, n_reps + 1):
        true_theta = (
            base_theta +
            rng.normal(0.0, theta_sd, size=base_theta.size)
        )
        true_parameters = decode_np(true_theta, "D")

        synthetic_groups = simulate_groups_from_parameters(
            template_groups,
            true_parameters,
            rng,
        )
        synthetic_counts = stack_group_counts(synthetic_groups)

        # No warm start at the true parameter values.
        result, recovered = fit_model_cached(
            model="D",
            counts=synthetic_counts,
            n_starts=n_starts,
            seed=seed + rep * 100,
            warm_starts=None,
        )

        for parameter in PARAMETER_NAMES:
            rows.append({
                "rep": rep,
                "parameter": parameter,
                "true": true_parameters[parameter],
                "recovered": recovered[parameter],
                "error": (
                    recovered[parameter] -
                    true_parameters[parameter]
                ),
                "absolute_error": abs(
                    recovered[parameter] -
                    true_parameters[parameter]
                ),
                "optimizer_success": bool(result.success),
            })

    raw = pd.DataFrame(rows)

    summary_rows = []
    for parameter, dat in raw.groupby("parameter"):
        if dat["true"].nunique() > 1:
            correlation = dat["true"].corr(dat["recovered"])
        else:
            correlation = np.nan

        summary_rows.append({
            "parameter": parameter,
            "correlation": correlation,
            "bias": dat["error"].mean(),
            "MAE": dat["absolute_error"].mean(),
            "RMSE": np.sqrt(np.mean(dat["error"] ** 2)),
        })

    summary = pd.DataFrame(summary_rows)

    return raw, summary


parameter_recovery, parameter_recovery_summary = run_parameter_recovery_D(
    template_groups=groups,
    n_reps=PARAM_RECOVERY_REPS,
    n_starts=VALIDATION_STARTS,
    theta_sd=PARAM_RECOVERY_THETA_SD,
    seed=VALIDATION_SEED + 2000,
)

print("MODEL D PARAMETER RECOVERY")
display(parameter_recovery_summary)

parameter_recovery.to_csv(
    output_dir / "model_D_parameter_recovery_raw.csv",
    index=False,
)
parameter_recovery_summary.to_csv(
    output_dir / "model_D_parameter_recovery_summary.csv",
    index=False,
)

# Probability-like parameters.
fig, ax = plt.subplots(figsize=(6.0, 5.2))
ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1.2, label="Perfect recovery")

for parameter in [p for p in PARAMETER_NAMES if p != "epsilon"]:
    dat = parameter_recovery.loc[
        parameter_recovery["parameter"] == parameter
    ]
    ax.scatter(
        dat["true"],
        dat["recovered"],
        s=28,
        alpha=0.7,
        label=parameter,
    )

ax.set(
    xlabel="True value",
    ylabel="Recovered value",
    xlim=(0, 1),
    ylim=(0, 1),
    title="Model D parameter recovery",
)
ax.legend(frameon=False, ncol=2)
fig.tight_layout()
fig.savefig(
    output_dir / "model_D_parameter_recovery.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

# Epsilon on its own scale.
eps_dat = parameter_recovery.loc[
    parameter_recovery["parameter"] == "epsilon"
]

fig, ax = plt.subplots(figsize=(5.0, 4.4))
limit = max(
    eps_dat["true"].max(),
    eps_dat["recovered"].max(),
) * 1.05
ax.plot([0, limit], [0, limit], linestyle="--", linewidth=1.2)
ax.scatter(
    eps_dat["true"],
    eps_dat["recovered"],
    s=30,
    alpha=0.75,
)
ax.set(
    xlabel="True epsilon",
    ylabel="Recovered epsilon",
    xlim=(0, limit),
    ylim=(0, limit),
    title="Lapse parameter recovery",
)
fig.tight_layout()
fig.savefig(
    output_dir / "model_D_epsilon_recovery.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


## A–D model recovery

Synthetic datasets are generated from each model using that model's empirical fitted parameters and the observed group compositions. Every synthetic dataset is then fit with all four candidate models.

The confusion matrices report the fraction of datasets generated by each model that are selected by AIC or BIC. Strong diagonal mass means the model family is distinguishable under the current design.


In [ ]:
def run_model_recovery(
    template_groups,
    empirical_fits,
    n_reps=15,
    n_starts=5,
    seed=20260818,
):
    rng = np.random.default_rng(seed)
    n_observations = (
        len(template_groups) *
        GROUP_SIZE *
        len(ROUNDS)
    )

    rows = []

    for generating_model in MODELS:
        generating_parameters = empirical_fits[generating_model][1]

        for rep in range(1, n_reps + 1):
            synthetic_groups = simulate_groups_from_parameters(
                template_groups,
                generating_parameters,
                rng,
            )
            synthetic_counts = stack_group_counts(synthetic_groups)

            candidate_fits = fit_all_models_validation(
                synthetic_counts,
                n_starts=n_starts,
                seed=(
                    seed +
                    100000 * MODELS.index(generating_model) +
                    rep * 100
                ),
            )

            metrics = []

            for candidate_model in MODELS:
                result, _ = candidate_fits[candidate_model]
                k = N_PARAMETERS[candidate_model]

                metrics.append({
                    "candidate_model": candidate_model,
                    "NLL": float(result.fun),
                    "AIC": float(2 * result.fun + 2 * k),
                    "BIC": float(
                        2 * result.fun +
                        k * np.log(n_observations)
                    ),
                    "optimizer_success": bool(result.success),
                })

            winner_AIC = min(
                metrics,
                key=lambda x: x["AIC"],
            )["candidate_model"]

            winner_BIC = min(
                metrics,
                key=lambda x: x["BIC"],
            )["candidate_model"]

            for metric in metrics:
                rows.append({
                    "generating_model": generating_model,
                    "rep": rep,
                    **metric,
                    "winner_AIC": winner_AIC,
                    "winner_BIC": winner_BIC,
                })

    raw = pd.DataFrame(rows)

    unique_datasets = raw[
        [
            "generating_model",
            "rep",
            "winner_AIC",
            "winner_BIC",
        ]
    ].drop_duplicates()

    confusion_AIC = pd.crosstab(
        unique_datasets["generating_model"],
        unique_datasets["winner_AIC"],
        normalize="index",
    ).reindex(
        index=MODELS,
        columns=MODELS,
        fill_value=0,
    )

    confusion_BIC = pd.crosstab(
        unique_datasets["generating_model"],
        unique_datasets["winner_BIC"],
        normalize="index",
    ).reindex(
        index=MODELS,
        columns=MODELS,
        fill_value=0,
    )

    return raw, confusion_AIC, confusion_BIC


model_recovery, recovery_AIC, recovery_BIC = run_model_recovery(
    template_groups=groups,
    empirical_fits=fits,
    n_reps=MODEL_RECOVERY_REPS,
    n_starts=VALIDATION_STARTS,
    seed=VALIDATION_SEED + 3000,
)

print("MODEL RECOVERY — AIC")
display(recovery_AIC)

print("MODEL RECOVERY — BIC")
display(recovery_BIC)

model_recovery.to_csv(
    output_dir / "model_recovery_raw.csv",
    index=False,
)
recovery_AIC.to_csv(
    output_dir / "model_recovery_AIC.csv",
)
recovery_BIC.to_csv(
    output_dir / "model_recovery_BIC.csv",
)

for criterion, matrix in [
    ("AIC", recovery_AIC),
    ("BIC", recovery_BIC),
]:
    fig, ax = plt.subplots(figsize=(5.2, 4.5))

    image = ax.imshow(
        matrix.to_numpy(),
        vmin=0,
        vmax=1,
        aspect="equal",
    )

    for row in range(len(MODELS)):
        for col in range(len(MODELS)):
            ax.text(
                col,
                row,
                f"{matrix.iloc[row, col]:.2f}",
                ha="center",
                va="center",
            )

    ax.set_xticks(range(len(MODELS)), MODELS)
    ax.set_yticks(range(len(MODELS)), MODELS)
    ax.set(
        xlabel="Selected model",
        ylabel="Generating model",
        title=f"Model recovery ({criterion})",
    )

    fig.colorbar(image, ax=ax, label="Selection proportion")
    fig.tight_layout()
    fig.savefig(
        output_dir / f"model_recovery_{criterion}.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()


## Validation outputs

The cells above save all raw and summarized validation results into the same `python_model_outputs` directory as the main model fit. For the paper, inspect the recovery diagnostics before interpreting the fitted human/AI parameters as identified mechanisms.
